In [1]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path.cwd().parent))

from generate_PhenomTE import *

%matplotlib inline

2.1.0


In [2]:
help(gwe.measure_eccentricity)

Help on function measure_eccentricity in module gw_eccentricity.gw_eccentricity:

measure_eccentricity(tref_in=None, fref_in=None, method='Amplitude', dataDict=None, num_orbits_to_exclude_before_merger=2, precessing=False, frame='inertial', debug_level=0, extra_kwargs=None)
    Measure eccentricity and mean anomaly from a gravitational waveform.
    
    Eccentricity is measured using the GW frequency omega_gw(t) =
    d(phase_gw)/dt. Throughout this documentation, we will refer to phase_gw,
    omega_gw and amp_gw. For nonprecessing systems, these quantities are simply
    the corresponding values of the (2, 2) mode,
    
    amp_gw = amp22, phase_gw = phase22 and omega_gw = omega22.
    
    On the other hand, for precessing systems, we use Eq.(48) and (49) of
    arXiv:1701.00550 to define amp_gw and phase_gw. amp_gw [phase_gw] is
    defined using a symmetric [antisymmetric] combination of
    amplitude [phase] of (2, 2) and (2, -2) mode in the coprecessing frame,
    
    amp_gw =

In [3]:
    
# Sampling parameters
sampling_frequency = 2048 # or 4096
duration = 2 # seconds
time_array = np.linspace(-duration, 0, int(sampling_frequency * duration))  # time in seconds

# Define parameter sets to compare
parameter_sets = [
    # {"ecc_ref": 0.0, "mass_ratio": 1.0, "chi1": 0.0, "chi2": 0.0, "label": "Circular, q=1"},
    {"ecc_ref": 0.1, "mass_ratio": 1.0, "chi1": 0.0, "chi2": 0.0, "label": "e=0.1, q=1"},
    {"ecc_ref": 0.3, "mass_ratio": 1.0, "chi1": 0.0, "chi2": 0.0, "label": "e=0.3, q=1"},
    {"ecc_ref": 0.2, "mass_ratio": 5.0, "chi1": 0.0, "chi2": 0.0, "label": "e=0.2, q=5"},
    {"ecc_ref": 0.2, "mass_ratio": 1.0, "chi1": 0.5, "chi2": -0.3, "label": "e=0.2, χ₁=0.5"},
    {"ecc_ref": 0.2, "mass_ratio": 10.0, "chi1": 0.8, "chi2": -0.5, "label": "e=0.2, q=10, χ₁=0.8"},
]

print("\n" + "="*60)
print("MULTI-WAVEFORM COMPARISON ANALYSIS (PHASE + AMPLITUDE)")
print("="*60 + "\n")

# Store results
mapping_results = []
phase_L_results = []
phase_t_results = []
amp_L_results = []
amp_t_results = []
waveform_lengths = {}

for params in parameter_sets:
    print(f"\nProcessing: {params['label']}")
    
    # Create waveform instance with these parameters
    wp_compare = Waveform_Properties(
        time_array=time_array.copy(),
        mass_ratio=params["mass_ratio"],
        chi1=params["chi1"],
        chi2=params["chi2"],
        ecc_ref=params["ecc_ref"],
        f_ref=20,
        f_lower=10,
        phiRef=0,
        inclination=0,
        mean_anomaly_ref=0,
        parametrization='time',  # We'll convert manually to L domain
        truncate_at_ISCO=False,
        truncate_at_tmin=True
    )
    
    # Simulate waveform
    hp, hc, time_arr = wp_compare.simulate_waveform_l(
        truncate_at_tmin=True,
        truncate_at_ISCO=False,
        update_results=True
    )

    # Store temporarily
    waveform_lengths[params["label"]] = {
        'n_points': len(time_arr),
        'time_min': time_arr.min(),
        'time_max': time_arr.max(),
        'hp': hp,
        'hc': hc,
        'time_arr': time_arr,
        'params': params
    }

  
    # Compute L-domain mapping for PHASE
    l_domain_phase = wp_compare.to_l_domain(
        property_name='phase',
        make_diagnostic_plots=False,
        plot_in_L_domain=False,
        plot_mapping=True,
        save_fig=False
    )

    print(wp_compare.mean_anomaly[:5])
    plt.show()
    
    
    # Compute L-domain mapping for AMPLITUDE
    l_domain_amp = wp_compare.to_l_domain(
        property_name='amplitude',
        make_diagnostic_plots=False,
        plot_in_L_domain=False,
        save_fig=False
    )
    
    # Store mapping results (same for both properties since mapping is waveform-based)
    mapping_results.append({
        'label': params["label"],
        'ecc_ref': params["ecc_ref"],
        'mass_ratio': params["mass_ratio"],
        'chi1': params["chi1"],
        'chi2': params["chi2"],
        'mean_anomaly': l_domain_phase['mean_anomaly'],
        'tref_out': l_domain_phase['tref_out'],
        'time_to_mean_anomaly': l_domain_phase['time_to_mean_anomaly'],
        'mean_anomaly_to_time': l_domain_phase['mean_anomaly_to_time'],
    })
    
    # Store phase results
    phase_L_results.append({
        'label': params["label"],
        'L_out': l_domain_phase['mean_anomaly'],
        'phase_in_L': l_domain_phase['phase_in_L'],
    })
    
    # Store amplitude results
    amp_L_results.append({
        'label': params["label"],
        'L_out': l_domain_amp['mean_anomaly'],
        'amp_in_L': l_domain_amp['amplitude_in_L'],
    })
    
    # Also get properties in time domain for comparison
    phase_t = wp_compare.phase(hp, hc)
    amp_t = wp_compare.amplitude(hp, hc, geometric_units=True)
    
    phase_t_results.append({
        'label': params["label"],
        'time': time_arr,
        'phase': phase_t,
    })
    
    amp_t_results.append({
        'label': params["label"],
        'time': time_arr,
        'amplitude': amp_t,
    })
# Find minimum length
min_length = min(wf['n_points'] for wf in waveform_lengths.values())
min_length_label = [k for k, v in waveform_lengths.items() if v['n_points'] == min_length][0]





print("\n" + "="*60)
print("PLOTTING COMPARISONS")
print("="*60 + "\n")

# Ensure output directory exists
os.makedirs('Images/Validation', exist_ok=True)

# ========================================================================
# FIGURE 1: Time vs Mean Anomaly mappings for all waveforms
# ========================================================================
fig_mappings, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(mapping_results)))

for i, result in enumerate(mapping_results):
    color = colors[i]
    label = result['label']
    
    # Plot t → ℓ mapping
    axes[0].plot(result['tref_out'], result['mean_anomaly'], 
                 color=color, linewidth=1.2, alpha=0.9, label=label)

axes[0].set_xlabel('Time t [M]', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
axes[0].set_title('(a) Time ↔ Mean Anomaly Mapping Comparison', fontsize=13, fontweight='bold')
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot ℓ → t mapping (inverse)
for i, result in enumerate(mapping_results):
    color = colors[i]
    label = result['label']
    
    axes[1].plot(result['mean_anomaly'], result['tref_out'], 
                 color=color, linewidth=1.2, alpha=0.9, label=label)

axes[1].set_xlabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Time t [M]', fontsize=12, fontweight='bold')
axes[1].set_title('(b) Mean Anomaly → Time (Inverse Mapping)', fontsize=13, fontweight='bold')
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(True, alpha=0.3)

fig_mappings.suptitle('Waveform Parameter Dependence: t ↔ ℓ Mappings', 
                      fontsize=14, fontweight='bold', y=1.02)
fig_mappings.tight_layout()

figname = 'Images/Validation/t_vs_L_mappings_comparison.png'
fig_mappings.savefig(figname, dpi=300, bbox_inches='tight')
print(f"\nFigure saved: {figname}")

# ========================================================================
# FIGURE 2: Phases vs Mean Anomaly (L domain) - ALL WAVEFORMS
# ========================================================================
fig_phase_L, ax_L = plt.subplots(figsize=(14, 6))

for i, result in enumerate(phase_L_results):
    color = colors[i]
    label = result['label']
    
    cut_idx = len(result) - min_length
    phase_L_result = result['phase_in_L']
    L_out = result['L_out']

    ax_L.plot(L_out, phase_L_result, 
              color=color, linewidth=1.2, alpha=0.9, label=label)

ax_L.set_xlabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
ax_L.set_ylabel('Phase φ [rad]', fontsize=12, fontweight='bold')
ax_L.set_title('Phase vs Mean Anomaly (L-domain Representation)', 
               fontsize=13, fontweight='bold')
ax_L.legend(loc='best', fontsize=9)
ax_L.grid(True, alpha=0.3)

fig_phase_L.tight_layout()

figname_L_phase = 'Images/Validation/phases_vs_L_comparison.png'
fig_phase_L.savefig(figname_L_phase, dpi=300, bbox_inches='tight')
print(f"Figure saved: {figname_L_phase}")

# ========================================================================
# FIGURE 3: Amplitudes vs Mean Anomaly (L domain) - ALL WAVEFORMS
# ========================================================================
fig_amp_L, ax_L_amp = plt.subplots(figsize=(14, 6))

for i, result in enumerate(amp_L_results):
    color = colors[i]
    label = result['label']
    
    ax_L_amp.plot(result['L_out'], result['amp_in_L'], 
                  color=color, linewidth=1.2, alpha=0.9, label=label)

ax_L_amp.set_xlabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
ax_L_amp.set_ylabel('Amplitude A [geom units]', fontsize=12, fontweight='bold')
ax_L_amp.set_title('Amplitude vs Mean Anomaly (L-domain Representation)', 
                   fontsize=13, fontweight='bold')
ax_L_amp.legend(loc='best', fontsize=9)
ax_L_amp.grid(True, alpha=0.3)

fig_amp_L.tight_layout()

figname_L_amp = 'Images/Validation/amplitudes_vs_L_comparison.png'
fig_amp_L.savefig(figname_L_amp, dpi=300, bbox_inches='tight')
print(f"Figure saved: {figname_L_amp}")

# ========================================================================
# FIGURE 4: Phases vs Time (T domain) - ALL WAVEFORMS
# ========================================================================
fig_phase_T, ax_T = plt.subplots(figsize=(14, 6))

for i, result in enumerate(phase_t_results):
    color = colors[i]
    label = result['label']
    
    ax_T.plot(result['time'], result['phase'], 
              color=color, linewidth=1.2, alpha=0.9, label=label)

ax_T.set_xlabel('Time t [M]', fontsize=12, fontweight='bold')
ax_T.set_ylabel('Phase φ [rad]', fontsize=12, fontweight='bold')
ax_T.set_title('Phase vs Time (Time-domain Representation)', 
               fontsize=13, fontweight='bold')
ax_T.legend(loc='best', fontsize=9)
ax_T.grid(True, alpha=0.3)

fig_phase_T.tight_layout()

figname_T_phase = 'Images/Validation/phases_vs_T_comparison.png'
fig_phase_T.savefig(figname_T_phase, dpi=300, bbox_inches='tight')
print(f"Figure saved: {figname_T_phase}")

# ========================================================================
# FIGURE 5: Amplitudes vs Time (T domain) - ALL WAVEFORMS
# ========================================================================
fig_amp_T, ax_T_amp = plt.subplots(figsize=(14, 6))

for i, result in enumerate(amp_t_results):
    color = colors[i]
    label = result['label']
    
    ax_T_amp.plot(result['time'], result['amplitude'], 
                  color=color, linewidth=1.2, alpha=0.9, label=label)

ax_T_amp.set_xlabel('Time t [M]', fontsize=12, fontweight='bold')
ax_T_amp.set_ylabel('Amplitude A [geom units]', fontsize=12, fontweight='bold')
ax_T_amp.set_title('Amplitude vs Time (Time-domain Representation)', 
                   fontsize=13, fontweight='bold')
ax_T_amp.legend(loc='best', fontsize=9)
ax_T_amp.grid(True, alpha=0.3)

fig_amp_T.tight_layout()

figname_T_amp = 'Images/Validation/amplitudes_vs_T_comparison.png'
fig_amp_T.savefig(figname_T_amp, dpi=300, bbox_inches='tight')
print(f"Figure saved: {figname_T_amp}")

# ========================================================================
# FIGURE 6: Side-by-side for PHASE (T vs L domain)
# ========================================================================
if len(phase_t_results) > 0:
    idx = 0  # First waveform
    ref_label = phase_t_results[idx]['label']
    
    fig_side_by_side_phase, (ax_T_single, ax_L_single) = plt.subplots(
        1, 2, figsize=(14, 6), sharey=True
    )
    
    # Left: Time domain
    ax_T_single.plot(phase_t_results[idx]['time'], 
                     phase_t_results[idx]['phase'], 
                     color='blue', linewidth=1.5, label=ref_label)
    ax_T_single.set_xlabel('Time t [M]', fontsize=12, fontweight='bold')
    ax_T_single.set_ylabel('Phase φ [rad]', fontsize=12, fontweight='bold')
    ax_T_single.set_title(f'(a) Time Domain: {ref_label}', fontsize=12)
    ax_T_single.legend(loc='best', fontsize=9)
    ax_T_single.grid(True, alpha=0.3)
    
    # Right: L domain
    ax_L_single.plot(phase_L_results[idx]['L_out'], 
                     phase_L_results[idx]['phase_in_L'], 
                     color='red', linewidth=1.5, label=ref_label)
    ax_L_single.set_xlabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
    ax_L_single.set_title(f'(b) L-domain: {ref_label}', fontsize=12)
    ax_L_single.legend(loc='best', fontsize=9)
    ax_L_single.grid(True, alpha=0.3)
    
    fig_side_by_side_phase.suptitle(f'Phase: Side-by-Side Comparison (Time vs L Domain)\n{ref_label}', 
                                    fontsize=13, fontweight='bold', y=1.02)
    fig_side_by_side_phase.tight_layout()
    
    figname_sb_phase = 'Images/Validation/T_vs_L_side_by_side_phase.png'
    fig_side_by_side_phase.savefig(figname_sb_phase, dpi=300, bbox_inches='tight')
    print(f"Figure saved: {figname_sb_phase}")

# ========================================================================
# FIGURE 7: Side-by-side for AMPLITUDE (T vs L domain)
# ========================================================================
if len(amp_t_results) > 0:
    idx = 0  # First waveform
    ref_label = amp_t_results[idx]['label']
    
    fig_side_by_side_amp, (ax_T_single_amp, ax_L_single_amp) = plt.subplots(
        1, 2, figsize=(14, 6), sharey=True
    )
    
    # Left: Time domain
    ax_T_single_amp.plot(amp_t_results[idx]['time'], 
                         amp_t_results[idx]['amplitude'], 
                         color='blue', linewidth=1.5, label=ref_label)
    ax_T_single_amp.set_xlabel('Time t [M]', fontsize=12, fontweight='bold')
    ax_T_single_amp.set_ylabel('Amplitude A', fontsize=12, fontweight='bold')
    ax_T_single_amp.set_title(f'(a) Time Domain: {ref_label}', fontsize=12)
    ax_T_single_amp.legend(loc='best', fontsize=9)
    ax_T_single_amp.grid(True, alpha=0.3)
    
    # Right: L domain
    ax_L_single_amp.plot(amp_L_results[idx]['L_out'], 
                         amp_L_results[idx]['amp_in_L'], 
                         color='red', linewidth=1.5, label=ref_label)
    ax_L_single_amp.set_xlabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
    ax_L_single_amp.set_title(f'(b) L-domain: {ref_label}', fontsize=12)
    ax_L_single_amp.legend(loc='best', fontsize=9)
    ax_L_single_amp.grid(True, alpha=0.3)
    
    fig_side_by_side_amp.suptitle(f'Amplitude: Side-by-Side Comparison (Time vs L Domain)\n{ref_label}', 
                                  fontsize=13, fontweight='bold', y=1.02)
    fig_side_by_side_amp.tight_layout()
    
    figname_sb_amp = 'Images/Validation/T_vs_L_side_by_side_amplitude.png'
    fig_side_by_side_amp.savefig(figname_sb_amp, dpi=300, bbox_inches='tight')
    print(f"Figure saved: {figname_sb_amp}")

# ========================================================================
# FIGURE 8: Summary grid - all phases in L-domain
# ========================================================================
n_waveforms = len(phase_L_results)
cols = min(3, n_waveforms)
rows = (n_waveforms + cols - 1) // cols

fig_grid_phase, axes_grid_phase = plt.subplots(rows, cols, figsize=(5*cols, 4*rows), 
                                                sharex=True, sharey=True)

if rows == 1:
    axes_grid_phase = axes_grid_phase.reshape(1, -1)

for i, result in enumerate(phase_L_results):
    row = i // cols
    col = i % cols
    
    ax = axes_grid_phase[row, col]
    ax.plot(result['L_out'], result['phase_in_L'], 
            color=colors[i], linewidth=1.2)
    
    ax.set_title(f"{result['label']}", fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=8)

# Hide unused subplots
for i in range(n_waveforms, rows * cols):
    row = i // cols
    col = i % cols
    fig_grid_phase.delaxes(axes_grid_phase[row, col])

fig_grid_phase.supxlabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
fig_grid_phase.supylabel('Phase φ [rad]', fontsize=12, fontweight='bold')
fig_grid_phase.suptitle('Phase vs Mean Anomaly Across All Waveform Parameters', 
                        fontsize=13, fontweight='bold', y=1.02)
fig_grid_phase.tight_layout()

figname_grid_phase = 'Images/Validation/all_phases_L_domain_grid.png'
fig_grid_phase.savefig(figname_grid_phase, dpi=300, bbox_inches='tight')
print(f"Figure saved: {figname_grid_phase}")

# ========================================================================
# FIGURE 9: Summary grid - all amplitudes in L-domain
# ========================================================================
fig_grid_amp, axes_grid_amp = plt.subplots(rows, cols, figsize=(5*cols, 4*rows), 
                                            sharex=True, sharey=True)

if rows == 1:
    axes_grid_amp = axes_grid_amp.reshape(1, -1)

for i, result in enumerate(amp_L_results):
    row = i // cols
    col = i % cols
    
    ax = axes_grid_amp[row, col]
    ax.plot(result['L_out'], result['amp_in_L'], 
            color=colors[i], linewidth=1.2)
    
    ax.set_title(f"{result['label']}", fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=8)

# Hide unused subplots
for i in range(n_waveforms, rows * cols):
    row = i // cols
    col = i % cols
    fig_grid_amp.delaxes(axes_grid_amp[row, col])

fig_grid_amp.supxlabel('Mean Anomaly ℓ [rad]', fontsize=12, fontweight='bold')
fig_grid_amp.supylabel('Amplitude A', fontsize=12, fontweight='bold')
fig_grid_amp.suptitle('Amplitude vs Mean Anomaly Across All Waveform Parameters', 
                      fontsize=13, fontweight='bold', y=1.02)
fig_grid_amp.tight_layout()

figname_grid_amp = 'Images/Validation/all_amplitudes_L_domain_grid.png'
fig_grid_amp.savefig(figname_grid_amp, dpi=300, bbox_inches='tight')
print(f"Figure saved: {figname_grid_amp}")

# ========================================================================
# FIGURE 10: Mappings normalized (offset for clarity)
# ========================================================================
fig_normalized, ax_norm = plt.subplots(figsize=(14, 6))

offsets = np.linspace(0, 5*n_waveforms, n_waveforms)

for i, result in enumerate(mapping_results):
    color = colors[i]
    label = result['label']
    
    # Offset the curve vertically for clarity
    offset_t = offsets[i]
    
    ax_norm.plot(result['tref_out'], 
                 result['mean_anomaly'] + offset_t,
                 color=color, linewidth=1.2, alpha=0.9)
    
    # Add label near the curve
    ax_norm.text(result['tref_out'].mean(), 
                 result['mean_anomaly'].mean() + offset_t,
                 label, fontsize=8, color=color,
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, ec=color))

ax_norm.set_xlabel('Time t [M]', fontsize=12, fontweight='bold')
ax_norm.set_ylabel('Mean Anomaly ℓ [rad] (offset for clarity)', 
                   fontsize=12, fontweight='bold')
ax_norm.set_title('Normalized Time ↔ Mean Anomaly Mappings', 
                  fontsize=13, fontweight='bold')
ax_norm.grid(True, alpha=0.3)

fig_normalized.tight_layout()

figname_norm = 'Images/Validation/mappings_normalized.png'
fig_normalized.savefig(figname_norm, dpi=300, bbox_inches='tight')
print(f"Figure saved: {figname_norm}")


MULTI-WAVEFORM COMPARISON ANALYSIS (PHASE + AMPLITUDE)


Processing: e=0.1, q=1
time : SimInspiral_M_independent e = 0.1, l=0, q=1.0, chi1=0.0, chi2=0.0, len = 4296, M = None, lum_dist=None, t=[-6932, 165, num=4296], f_lower=9.999999999999998, f_ref=19.999999999999996 | computation time = 4.304 seconds
time : SimInspiral_M_independent e = 0, l=0, q=1.0, chi1=0.0, chi2=0.0, len = 4316, M = None, lum_dist=None, t=[-6949, 181, num=4316], f_lower=9.999999999999998, f_ref=19.999999999999996 | computation time = 0.022 seconds
t_l: 4096 4296 4316
orbital
Help on function measure_eccentricity in module gw_eccentricity.gw_eccentricity:

measure_eccentricity(tref_in=None, fref_in=None, method='Amplitude', dataDict=None, num_orbits_to_exclude_before_merger=2, precessing=False, frame='inertial', debug_level=0, extra_kwargs=None)
    Measure eccentricity and mean anomaly from a gravitational waveform.
    
    Eccentricity is measured using the GW frequency omega_gw(t) =
    d(phase_gw)/dt. Throug

TypeError: measure_eccentricity() got an unexpected keyword argument 'number_of_samples'